# Colab setup
Run the cells in order. Set BRANCH in the clone cell to the branch you want to run. Rerunning that cell fetches the selected branch, updates it, and checks that the required project files exist before installation.

## 1. Check GPU availability

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Select a branch and clone or update the GitHub repository

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/tiendinhquang2104/multimodal-churn-fusion.git'
BRANCH = 'feature/dev'  # Change this to another pushed branch when needed.
PROJECT_DIR = Path('/content/multimodal-churn-fusion')

if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    local_branch = subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'show-ref', '--verify', '--quiet', f'refs/heads/{BRANCH}']
    )
    if local_branch.returncode == 0:
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'switch', BRANCH], check=True)
    else:
        subprocess.run(
            ['git', '-C', str(PROJECT_DIR), 'switch', '--track', '-c', BRANCH, f'origin/{BRANCH}'],
            check=True,
        )
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
elif PROJECT_DIR.exists():
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a Git checkout. Inspect it before cloning.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

required = ['pyproject.toml', 'requirements.txt', 'configs/baseline.yaml', 'src/multimodal_churn']
missing = [name for name in required if not (PROJECT_DIR / name).exists()]
if missing:
    raise RuntimeError(
        'The GitHub checkout is missing project files: ' + ', '.join(missing)
        + f'. Check that branch {BRANCH!r} contains the project, then rerun this cell.'
    )
print('Repository ready:', PROJECT_DIR, 'branch:', BRANCH)

## 3. Move into the verified project

In [ ]:
%cd /content/multimodal-churn-fusion

## 4. Install runtime requirements

In [ ]:
%pip install -r requirements.txt

## 5. Install the package in editable mode

In [ ]:
%pip install -e .

## 6. Mount Google Drive
Authorize the Google account that can open the H&M shared folder. If the folder appears only in Shared with me, add a shortcut to My Drive. The folder URL is not a Colab filesystem path. An "already mounted" message is normal.

In [ ]:
from pathlib import Path
from google.colab import drive

try:
    drive.mount('/content/drive', timeout_ms=300000)
except ValueError as exc:
    raise RuntimeError(
        'Google Drive mount failed before the dataset could be read. Reconnect the Colab runtime, '
        'rerun this cell, and authorize the Google account that can open the H&M folder.'
    ) from exc
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Drive mount completed, but /content/drive/MyDrive is unavailable.')

## 7. Locate the shared dataset and output paths
The H&M folder is 02_hm_fashion and contains the 30.8 GB ZIP. This cell checks the thesis path and common My Drive shortcut locations. Set DATASETS_ROOT_OVERRIDE to the parent of 02_hm_fashion if your shortcut is elsewhere. Future datasets use configs/datasets/<name>.yaml. The ZIP is read in place.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/content/multimodal-churn-fusion')
SRC_DIR = PROJECT_DIR / 'src'
if not (SRC_DIR / 'multimodal_churn').is_dir():
    raise FileNotFoundError('Project source is missing. Rerun the clone/update cell first.')
os.chdir(PROJECT_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from multimodal_churn.utils.config import find_drive_dataset_paths, load_config

baseline_path = Path('configs/baseline.yaml')
DATASET_NAME = load_config(baseline_path)['dataset']['name']
MY_DRIVE = Path('/content/drive/MyDrive')
DATASETS_ROOT_OVERRIDE = None  # Example: MY_DRIVE / 'My shortcuts' / 'Datasets'
ARTIFACTS_ROOT = MY_DRIVE / 'multimodal-churn-fusion'

dataset_config = Path('configs/datasets') / f'{DATASET_NAME}.yaml'
config = load_config(baseline_path, dataset_path=dataset_config)
try:
    DATASETS_ROOT, dataset_paths = find_drive_dataset_paths(
        config, MY_DRIVE, DATASETS_ROOT_OVERRIDE
    )
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f'{exc}\nMount the correct Google account, or add the shared folder as a shortcut '
        'to My Drive and set DATASETS_ROOT_OVERRIDE to its parent.'
    ) from exc
DATASET_PATH = dataset_paths['folder']
DATASET_ARCHIVE = dataset_paths.get('archive')
EMBEDDINGS_PATH = ARTIFACTS_ROOT / 'data' / 'embeddings' / DATASET_NAME
CHECKPOINTS_PATH = ARTIFACTS_ROOT / 'outputs' / 'checkpoints' / DATASET_NAME
for path in (EMBEDDINGS_PATH, CHECKPOINTS_PATH):
    path.mkdir(parents=True, exist_ok=True)

print('Dataset root:', DATASETS_ROOT)
print('Dataset folder:', DATASET_PATH)
print('Dataset archive:', DATASET_ARCHIVE)
print('Embeddings:', EMBEDDINGS_PATH)
print('Checkpoints:', CHECKPOINTS_PATH)

## 8. Verify package imports and configuration

In [ ]:
from multimodal_churn import MultimodalChurnModel
from multimodal_churn.models.encoders import TransactionEncoder, TextEncoder, ImageEncoder
from multimodal_churn.models.fusion import ConcatFusion
print('Package import OK; dataset:', config['dataset']['name'])
print('Embedding dimension:', config['model']['common_embedding_dim'])